# Go2 Gait Recording Visualizer
Plots IMU, joint states, foot forces, and power data from a `.jsonl` recording.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
JSONL_FILE = Path('record_gait_20260518_093912.jsonl')

records = []
with open(JSONL_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} records')

# --- flatten into arrays ---
t0 = records[0]['wall_time']
t  = np.array([r['wall_time'] - t0 for r in records])

imu_rpy  = np.array([r['imu_rpy']  for r in records])
imu_gyro = np.array([r['imu_gyro'] for r in records])
imu_acc  = np.array([r['imu_acc']  for r in records])

foot_force     = np.array([r['foot_force']     for r in records])
foot_force_est = np.array([r['foot_force_est'] for r in records])

power_v = np.array([r['power_v'] for r in records])
power_a = np.array([r['power_a'] for r in records])

# controller
AXES   = ['lx', 'rx', 'ry', 'ly']
BUTTONS = ['R1','L1','Start','Select','R2','L2','F1','F3','A','B','X','Y','Up','Right','Down','Left']
remote_axes    = np.array([[r['remote'][k]              for k in AXES]    for r in records])
remote_buttons = np.array([[r['remote']['buttons'][k]   for k in BUTTONS] for r in records], dtype=np.int8)

JOINT_NAMES = [j['name'] for j in records[0]['joints']]
N = len(records)
nj = len(JOINT_NAMES)

q       = np.zeros((N, nj))
dq      = np.zeros((N, nj))
tau_est = np.zeros((N, nj))
temp    = np.zeros((N, nj))

for i, r in enumerate(records):
    for j in r['joints']:
        idx = JOINT_NAMES.index(j['name'])
        q[i, idx]       = j['q']
        dq[i, idx]      = j['dq']
        tau_est[i, idx] = j['tau_est']
        temp[i, idx]    = j['temperature']

print('Duration: {:.1f} s  |  Avg rate: {:.1f} Hz'.format(t[-1], len(t)/t[-1]))

## 1 · IMU — Roll / Pitch / Yaw

In [ ]:
labels = ['Roll', 'Pitch', 'Yaw']
colors = ['tab:blue', 'tab:orange', 'tab:green']

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(t, np.degrees(imu_rpy[:, i]), color=colors[i], lw=0.8)
    ax.set_ylabel(f'{labels[i]} (deg)')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('IMU — Orientation (RPY)', fontweight='bold')
plt.tight_layout()
plt.show()

## 2 · IMU — Gyroscope & Accelerometer

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for i, label in enumerate(['x', 'y', 'z']):
    axes[0].plot(t, imu_gyro[:, i], lw=0.7, label=label)
    axes[1].plot(t, imu_acc[:, i],  lw=0.7, label=label)

axes[0].set_ylabel('Gyro (rad/s)')
axes[0].legend(loc='upper right', ncol=3)
axes[1].set_ylabel('Accel (m/s²)')
axes[1].legend(loc='upper right', ncol=3)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('IMU — Gyroscope & Accelerometer', fontweight='bold')
plt.tight_layout()
plt.show()

## 3 · Foot Forces

In [ ]:
FOOT_LABELS = ['FR', 'FL', 'RR', 'RL']
foot_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for i in range(4):
    axes[0].plot(t, foot_force[:, i],     lw=0.8, color=foot_colors[i], label=FOOT_LABELS[i])
    axes[1].plot(t, foot_force_est[:, i], lw=0.8, color=foot_colors[i], label=FOOT_LABELS[i])

axes[0].set_ylabel('Measured force')
axes[0].legend(ncol=4)
axes[1].set_ylabel('Estimated force')
axes[1].legend(ncol=4)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Foot Forces', fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Joint Positions (q)

In [ ]:
LEG_GROUPS = {
    'FR': [0, 1, 2],
    'FL': [3, 4, 5],
    'RR': [6, 7, 8],
    'RL': [9, 10, 11],
}
DOF_LABELS = ['Hip (0)', 'Thigh (1)', 'Calf (2)']
leg_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for leg_i, (leg, idxs) in enumerate(LEG_GROUPS.items()):
    for dof, (ax, idx) in enumerate(zip(axes, idxs)):
        ax.plot(t, np.degrees(q[:, idx]), lw=0.7, color=leg_colors[leg_i], label=leg)
        if leg_i == 0:
            ax.set_ylabel(f'{DOF_LABELS[dof]} (deg)')

axes[0].legend(ncol=4, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Joint Positions (q)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5 · Joint Velocities (dq)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for leg_i, (leg, idxs) in enumerate(LEG_GROUPS.items()):
    for dof, (ax, idx) in enumerate(zip(axes, idxs)):
        ax.plot(t, dq[:, idx], lw=0.7, color=leg_colors[leg_i], label=leg)
        if leg_i == 0:
            ax.set_ylabel(f'{DOF_LABELS[dof]} (rad/s)')

axes[0].legend(ncol=4, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Joint Velocities (dq)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6 · Joint Torques (τ estimated)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for leg_i, (leg, idxs) in enumerate(LEG_GROUPS.items()):
    for dof, (ax, idx) in enumerate(zip(axes, idxs)):
        ax.plot(t, tau_est[:, idx], lw=0.7, color=leg_colors[leg_i], label=leg)
        if leg_i == 0:
            ax.set_ylabel(f'{DOF_LABELS[dof]} (N·m)')

axes[0].legend(ncol=4, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Joint Torques (τ_est)', fontweight='bold')
plt.tight_layout()
plt.show()

## 7 · Motor Temperatures

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
cmap = plt.get_cmap('tab20', nj)
for i, name in enumerate(JOINT_NAMES):
    ax.plot(t, temp[:, i], lw=0.8, color=cmap(i), label=name)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature (°C)')
ax.legend(ncol=6, loc='upper right', fontsize=8)
ax.set_title('Motor Temperatures', fontweight='bold')
plt.tight_layout()
plt.show()

## 8 · Battery Power

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

axes[0].plot(t, power_v, color='tab:blue', lw=0.8)
axes[0].set_ylabel('Voltage (V)')

axes[1].plot(t, power_a, color='tab:orange', lw=0.8)
axes[1].set_ylabel('Current (A)')

axes[2].plot(t, power_v * power_a, color='tab:red', lw=0.8)
axes[2].set_ylabel('Power (W)')
axes[2].set_xlabel('Time (s)')

fig.suptitle('Battery / Power', fontweight='bold')
plt.tight_layout()
plt.show()

## 9 · Phase Portrait — Hip joints

In [ ]:
# Phase portrait: joint position vs velocity for each hip (index 0,3,6,9)
hip_idxs = [0, 3, 6, 9]
hip_names = ['FR_0', 'FL_0', 'RR_0', 'RL_0']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, idx, name, c in zip(axes.flat, hip_idxs, hip_names, leg_colors):
    sc = ax.scatter(np.degrees(q[:, idx]), dq[:, idx],
                    c=t, cmap='viridis', s=1, alpha=0.6)
    ax.set_xlabel('q (deg)')
    ax.set_ylabel('dq (rad/s)')
    ax.set_title(name)
    plt.colorbar(sc, ax=ax, label='time (s)')

fig.suptitle('Hip Phase Portraits (colour = time)', fontweight='bold')
plt.tight_layout()
plt.show()

## 10 · Controller — Joystick Axes

In [ ]:
AXIS_LABELS = {
    'lx': 'Left stick X  (strafe)',
    'ly': 'Left stick Y  (forward)',
    'rx': 'Right stick X (yaw)',
    'ry': 'Right stick Y (unused)',
}
axis_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for i, (ax, key) in enumerate(zip(axes, AXES)):
    ax.plot(t, remote_axes[:, i], color=axis_colors[i], lw=0.8)
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylabel(AXIS_LABELS[key], fontsize=9)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Controller — Joystick Axes', fontweight='bold')
plt.tight_layout()
plt.show()

## 11 · Controller — Button Presses

In [ ]:
# Only show buttons that were pressed at least once
active_mask = remote_buttons.any(axis=0)
active_btns = [b for b, m in zip(BUTTONS, active_mask) if m]
active_data = remote_buttons[:, active_mask]

if active_btns:
    fig, ax = plt.subplots(figsize=(14, max(3, len(active_btns) * 0.5)))
    cmap_btn = plt.get_cmap('tab20', len(active_btns))
    for i, name in enumerate(active_btns):
        # offset each button row for readability
        ax.fill_between(t, i + active_data[:, i], i, step='mid',
                        color=cmap_btn(i), alpha=0.8, label=name)
    ax.set_yticks(range(len(active_btns)))
    ax.set_yticklabels(active_btns)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Button')
    ax.set_title('Controller — Button Presses (active buttons only)', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.grid(axis='y', alpha=0)
    plt.tight_layout()
    plt.show()
else:
    print('No buttons were pressed during this recording.')